# 8.a — Ad hoc test: cross-fitted predictions and positive-unlabeled correction

**Ad hoc test, nothing of the current chain is changed.** It reads the final models of 3.g, the probabilities, novelty
and TMIN of 4.g and the soil of 2.g, and writes only in `<out_root>/test_crossfit_pu/<route>/<set>/` (fold models,
tables, maps). The current results stay as they are.

**Why.** The final map is predicted by the model fitted on all the training cells. On those cells it "knows the
answer": in sample only 0.3 % of the maize absences (cells with other crops and no maize) are called suitable, against
about 6 % out of sample (specificity 0.94, 5.h). And even without memorisation, a classifier trained on presences
against absences estimates "is the crop grown, given the climate", not "does the climate suit the crop": an absence
is often a land-use choice (another crop is grown there), not an unsuitable climate.

**Versions compared** (same rule otherwise: probability ≥ 0.5, not novel, growing season ≥ TMIN on the lower part of the
cell, soil ≥ SOIL_MIN; same novelty and TMIN as 4.g):
- **current**: the maps of 4.g / 6.b (final model everywhere, probability at equal class priors);
- **A, cross-fitted**: each training cell is predicted, in every climate, by the model of the fold of 3.g that did not
  see it (5 folds, seed 42, the cost and gamma of the final model); the other cells keep the final model. Removes the
  memorisation, keeps the meaning (equal priors);
- **A+B, cross-fitted + positive-unlabeled** (Elkan & Noto 2008): the presences are labelled, the cells with other
  crops are unlabeled (suitable or not). The Platt probability g = P(labelled | climate) of the SVM is that classifier;
  with c = the mean held-out g of the presences (from A), P(suitable | climate) = g / c (capped at 1). Assumption:
  where the climate suits the crop, whether it is grown does not depend on the climate.

In terms of the Platt probability g, the threshold 0.5 is g ≥ n1 / (n1 + n0) for the current and A versions (the share
of presences of the training table) and g ≥ 0.5 c for A+B: the tables print both.

**Compared**: on ERA5, the MIRCA2000 area and presence cells suitable, the absences called suitable, the suitable land
and the land north of 50° N outside the cropland; in the four runs, the hosing effect (cells assessable in both) and
the change of today's crop area suitable; maps (ERA5 vs MIRCA2000, 2050 hosing vs no hosing).

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time, hashlib
import numpy as np, pandas as pd, xarray as xr, joblib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from pathlib import Path
from joblib import Parallel, delayed
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()

# ---- parameters: the set of 3.g / 4.g / 5.h / 6.b
CHOSEN_VARIANT = "A_60d"
CHOSEN_RAD = os.environ.get("CHOSEN_RAD", "no_rad")
CELL_AREA_THR = 0.0005
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]
ERA5_NAME, ERA5_PERIOD = "ERA5", ("1981", "2014")
RUNS = ["gcm_2025_nohosing", "gcm_2025_hosing", "gcm_2050_nohosing", "gcm_2050_hosing"]
PAIRS = [("gcm_2025_hosing", "gcm_2025_nohosing"), ("gcm_2050_hosing", "gcm_2050_nohosing")]   # hosing - original
LEVELS = ["low", "high"]
N_FOLDS, SEED = 5, 42              # the folds of 3.g
N_JOBS = min(N_FOLDS, os.cpu_count() or 1)
OVERWRITE = False                  # False = fold models already fitted are read

SET = f"{CHOSEN_VARIANT}_{CHOSEN_RAD}"
TAG = f"{ERA5_NAME}_{ERA5_PERIOD[0]}_{ERA5_PERIOD[1]}_ecearth3"
ROUTE = cu.route(CELL_AREA_THR)
ERA5_DIR = cu.out_dir(CFG, "1_biovariables", f"{ERA5_NAME}_{ROUTE}", CHOSEN_VARIANT)
DELTA = cu.out_dir(CFG, "1_biovariables", f"GCM_delta_{ROUTE}", SET)
MODELS = cu.out_dir(CFG, "3_svm_train", f"{ERA5_NAME}_{ROUTE}", SET)
CLIM_DIR = cu.out_dir(CFG, "4_svm_predict", ROUTE, SET)
SOIL_DIR = cu.out_dir(CFG, "2_soil_suitability")
OUT = cu.out_dir(CFG, "test_crossfit_pu", ROUTE, SET)       # the only folder written
FOLD_DIR = OUT / "fold_models"; FOLD_DIR.mkdir(exist_ok=True)
SCEN = ["ERA5", *RUNS]

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
LAND_HA = np.nan_to_num(cu.land_area_ha(CFG))
LAND_MHA = LAND_HA * LAND / 1e6
ANY = cu.mirca_cropland(CFG).values                         # any MIRCA2000 cropland
LAT2 = np.broadcast_to(GRID["lat"][:, None], LAND.shape)
N50_OUT = LAND & ~ANY & (LAT2 > 50)
SOIL = {lv: xr.open_dataset(SOIL_DIR / f"soil_suitability_rf_{lv}_ecearth3.nc").load() for lv in LEVELS}
mha = lambda m: float(np.nansum(LAND_MHA * m))
print(f"set {SET} | models {MODELS} | 4.g {CLIM_DIR}\nwritten only in {OUT}")

## 1. Fold models (the folds of 3.g, the cost and gamma of the final model)

In [ ]:
def class_weight(y):
    return {1: 1.0, 0: (y == 1).sum() / (y == 0).sum()}      # as 3.g


def table_hash(X, y):
    return hashlib.md5(np.ascontiguousarray(X, dtype=np.float64).tobytes() + np.asarray(y, np.int8).tobytes()).hexdigest()


def fit_fold(X, y, tr, cost, gamma):
    s = StandardScaler().fit(X[tr])
    m = SVC(kernel="rbf", C=cost, gamma=gamma, class_weight=class_weight(y[tr]), probability=True,
            random_state=SEED).fit(s.transform(X[tr]), y[tr])
    return s, m


FOLDS = {}
for crop in CROPS:
    b = joblib.load(MODELS / f"svm_model_{TAG}_{crop}.joblib")
    df = pd.read_csv(ERA5_DIR / f"training_dataset_{TAG}_{crop}.csv")
    cols = b["bio_cols"]
    X, y = df[cols].values.astype(float), df["label"].values.astype(int)
    if b.get("table_hash"):
        assert b["table_hash"] == table_hash(X, y), f"{crop}: the training table changed since 3.g"
    path = FOLD_DIR / f"folds_{TAG}_{crop}.joblib"
    Fd = joblib.load(path) if path.exists() and not OVERWRITE else None
    if Fd is None or Fd["hash"] != table_hash(X, y) or Fd["cost"] != b["best_cost"] or Fd["gamma"] != b["best_gamma"]:
        t0 = time.time()
        folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
        fits = Parallel(n_jobs=N_JOBS)(delayed(fit_fold)(X, y, tr, b["best_cost"], b["best_gamma"]) for tr, _ in folds)
        Fd = {"fits": fits, "test_idx": [te for _, te in folds], "hash": table_hash(X, y),
              "cost": b["best_cost"], "gamma": b["best_gamma"]}
        joblib.dump(Fd, path)
        print(f"{crop}: 5 fold models fitted ({(time.time() - t0) / 60:.1f} min)", flush=True)
    else:
        print(f"{crop}: fold models read", flush=True)
    fold_of = np.empty(len(y), int)
    for k, te in enumerate(Fd["test_idx"]):
        fold_of[te] = k
    ii = np.searchsorted(GRID["lat"], df.lat.values - 1e-6); jj = np.searchsorted(GRID["lon"], df.lon.values - 1e-6)
    FOLDS[crop] = {"b": b, "cols": cols, "y": y, "fold_of": fold_of, "ii": ii, "jj": jj, "F": Fd}

## 2. The three versions on ERA5 and the four runs

In [ ]:
def bio_file(scenario, crop):
    if scenario == "ERA5":
        return ERA5_DIR / f"BIOvariables_{TAG}_{crop}.nc"
    return DELTA / f"BIOvariables_delta_{scenario}_{crop}.nc"


VERS = ["current", "A cross-fitted", "A+B cross-fitted + PU"]
era5_rows, amoc_rows, info_rows, MAPS = [], [], [], {}
for crop in CROPS:
    d = FOLDS[crop]; b = d["b"]; n1, n0 = b["n_class1"], b["n_class0"]
    ii, jj, y = d["ii"], d["jj"], d["y"]
    with xr.open_dataset(CLIM_DIR / f"suitability_{crop}.nc") as ds:
        P = {s: ds["suit_proba_platt"].sel(scenario=s).values.astype(float) for s in SCEN}
        NOV = {s: ds["novel"].sel(scenario=s).values == 1 for s in SCEN}
        COLD = {s: ds["below_tmin"].sel(scenario=s).values == 1 for s in SCEN}
        CUR = {s: ds["suit_bin"].sel(scenario=s).values == 1 for s in SCEN}
    G = {}                                                    # cross-fitted Platt probability at the training cells
    for s in SCEN:
        with xr.open_dataset(bio_file(s, crop)) as ds:
            Xs = cu.bio_array(ds, d["cols"], CFG)[ii, jj]
        g = np.full(len(y), np.nan)
        ok = np.isfinite(Xs).all(axis=1)
        for k, (sc, m) in enumerate(d["F"]["fits"]):
            sel = ok & (d["fold_of"] == k)
            if sel.any():
                g[sel] = m.predict_proba(sc.transform(Xs[sel]))[:, 1]
        G[s] = g
    c = float(np.nanmean(G["ERA5"][y == 1]))                  # Elkan & Noto: mean held-out probability of the presences
    clim = {v: {} for v in VERS}
    for s in SCEN:
        pa = P[s].copy()
        fin = np.isfinite(G[s])
        pa[ii[fin], jj[fin]] = G[s][fin]
        ok_s = LAND & np.isfinite(pa) & ~NOV[s] & ~COLD[s]
        clim["current"][s] = LAND & CUR[s]
        clim["A cross-fitted"][s] = ok_s & (cu.balanced_proba(pa, n1, n0) >= cu.CLIMATE_P)
        clim["A+B cross-fitted + PU"][s] = ok_s & (np.minimum(pa / c, 1.0) >= cu.CLIMATE_P)
    # check: outside the training cells, A must be the current map
    out_tr = LAND.copy(); out_tr[ii, jj] = False
    same = (clim["A cross-fitted"]["ERA5"] == clim["current"]["ERA5"])[out_tr].mean()
    info_rows.append({"crop": crop, "presences": n1, "absences": n0, "c (mean held-out g of the presences)": c,
                      "Platt threshold, current and A = n1/(n1+n0)": n1 / (n1 + n0), "Platt threshold, A+B = 0.5 c": 0.5 * c,
                      "A = current outside the training cells (%)": 100 * same})
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    pres_cells, abs_cells = (ii[y == 1], jj[y == 1]), (ii[y == 0], jj[y == 0])
    MAPS[crop] = {}
    for lv in LEVELS:
        ssr = np.where(LAND, np.nan_to_num(SOIL[lv][crop].values), 0.0)
        for v in VERS:
            S = {s: clim[v][s] & (ssr >= cu.SOIL_MIN) for s in SCEN}
            MAPS[crop][(v, lv)] = S
            e = S["ERA5"]
            era5_rows.append({"crop": crop, "level": lv, "version": v,
                              "MIRCA area suitable (%)": 100 * crop_ha[e].sum() / crop_ha.sum(),
                              "presence cells suitable (%)": 100 * e[pres_cells].mean(),
                              "absences (other crops) suitable (%)": 100 * e[abs_cells].mean(),
                              "suitable land (Mha)": mha(e), "N of 50, outside cropland (Mha)": mha(e & N50_OUT)})
            for h, o in PAIRS:
                both = LAND & ~(NOV[h] | NOV[o])
                ah, ao = mha(S[h] & both), mha(S[o] & both)
                amoc_rows.append({"crop": crop, "level": lv, "version": v, "pair": h.split("_")[1],
                                  "hosing effect (%)": 100 * (ah - ao) / ao if ao else np.nan,
                                  "today's crop area suitable, no hosing (%)": 100 * crop_ha[S[o]].sum() / crop_ha.sum(),
                                  "today's crop area: hosing effect (points)": 100 * (crop_ha[S[h]].sum() - crop_ha[S[o]].sum()) / crop_ha.sum()})
    print(f"{crop}: c = {c:.3f}", flush=True)
info = pd.DataFrame(info_rows).set_index("crop")
era5 = pd.DataFrame(era5_rows).set_index(["crop", "level", "version"])
amoc = pd.DataFrame(amoc_rows).set_index(["crop", "level", "pair", "version"])
info.to_csv(OUT / f"pu_constants_{SET}.csv"); era5.to_csv(OUT / f"era5_versions_{SET}.csv"); amoc.to_csv(OUT / f"amoc_versions_{SET}.csv")
with pd.option_context("display.max_rows", 300, "display.max_columns", 30, "display.width", 250):
    print("(0) constants"); display(info.round(3))
    print("(1) ERA5, final suitability (climate and soil)"); display(era5.round(2))
    print("(2) AMOC effect, cells assessable in both"); display(amoc["hosing effect (%)"].unstack("version")[VERS].round(2))
    print("(3) today's crop area: hosing effect (points)"); display(amoc["today's crop area: hosing effect (points)"].unstack("version")[VERS].round(2))
    print("(4) today's crop area suitable without hosing (%)"); display(amoc["today's crop area suitable, no hosing (%)"].unstack("version")[VERS].round(1))

## 3. Maps, high input: ERA5 vs MIRCA2000 and 2050 hosing vs no hosing, the three versions

In [ ]:
VS = (["#e6e6e6", "#1a9641", "#d7191c", "#a6d96a"], ["neither", "suitable and cultivated", "cultivated, not suitable", "suitable, not cultivated"])
CH = (["#e6e6e6", "#a6d96a", "#d7191c", "#1a9641"], ["not suitable in both", "suitable in both", "lost with the hosing", "gained with the hosing"])
DM = OUT / "maps"; DM.mkdir(exist_ok=True)
h, o = PAIRS[-1] if PAIRS else ("ERA5", "ERA5")
for crop in CROPS:
    crop_ha = np.nan_to_num(cu.crop_mask(CFG, crop)["area_ha"].values)
    pres = LAND & (crop_ha > CELL_AREA_THR * LAND_HA)
    fig, axes = plt.subplots(2, 3, figsize=(33, 11))
    for k, v in enumerate(VERS):
        S = MAPS[crop][(v, "high")]
        e = S["ERA5"]
        c0 = np.where(e & pres, 1.0, np.where(pres, 2.0, np.where(e, 3.0, 0.0)))
        c1 = np.where(S[o] & S[h], 1.0, np.where(S[o] & ~S[h], 2.0, np.where(~S[o] & S[h], 3.0, 0.0)))
        for ax, cc, (cols_, labs), t in ((axes[0, k], c0, VS, f"{crop}, {v}: ERA5 vs MIRCA2000 | suitable {mha(e):.0f} Mha"),
                                         (axes[1, k], c1, CH, f"{crop}, {v}: {h} vs {o}")):
            ax.pcolormesh(GRID["lon"], GRID["lat"], np.where(LAND, cc, np.nan), vmin=-0.5, vmax=3.5, cmap=ListedColormap(cols_), shading="nearest")
            ax.set_ylim(-60, 85); ax.set_title(t, fontsize=11)
            ax.legend(handles=[Patch(color=a, label=l) for a, l in zip(cols_, labs)], loc="lower left", fontsize=8, framealpha=0.9)
    fig.savefig(DM / f"{crop}_high_versions.png", dpi=90, bbox_inches="tight"); plt.show()
print("maps in", DM)